<a href="https://colab.research.google.com/github/Fahad-Alam-Jamal/Flyrank_ML_Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fahad-Alam-Jamal/Flyrank_ML_Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This notebook builds a small feature frame for the March 2026 refresh-opportunity lane and then tests a deliberate leakage trap.

## 1. Build the feature vector

The feature frame below uses one row per content item from the March 2026 performance slice and logs the heavy-tailed counts so the model sees a more stable scale.

In [1]:
import math
import os
import duckdb
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


def load_hf_token():
    from google.colab import userdata

    try:
        HF_TOKEN = userdata.get("HF_TOKEN")
        return HF_TOKEN
    except Exception:
        raise RuntimeError("HF_TOKEN not found in Colab Secrets.")

HF_TOKEN = load_hf_token()
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'
march_path = f'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'
april_path = f'{REL}/fact_content_daily_performance/month=2026-04/*.parquet'

feature_sql = f"""
WITH march_base AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions_march,
        SUM(gsc_clicks) AS gsc_clicks_march,
        AVG(gsc_avg_position) AS gsc_avg_position_march,
        SUM(ga4_sessions) AS ga4_sessions_march,
        MAX(CASE WHEN gsc_data_available THEN 1 ELSE 0 END) AS gsc_available_march,
        MAX(CASE WHEN ga4_data_available THEN 1 ELSE 0 END) AS ga4_available_march
    FROM read_parquet('{march_path}')
    GROUP BY 1, 2
),
april_base AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions_april
    FROM read_parquet('{april_path}')
    GROUP BY 1, 2
)
SELECT
    m.client_hash_id,
    m.content_hash_id,
    m.gsc_impressions_march,
    m.gsc_clicks_march,
    m.gsc_avg_position_march,
    m.ga4_sessions_march,
    m.gsc_available_march,
    m.ga4_available_march,
    CASE
        WHEN a.gsc_impressions_april IS NULL THEN NULL
        WHEN m.gsc_impressions_march = 0 THEN 0
        WHEN a.gsc_impressions_april < 0.8 * m.gsc_impressions_march THEN 1 ELSE 0
    END AS future_decline_proxy
FROM march_base m
LEFT JOIN april_base a USING (client_hash_id, content_hash_id)
"""

frame = con.execute(feature_sql).fetchdf()
frame = frame[frame['future_decline_proxy'].notna()].copy()
for col in ['gsc_impressions_march', 'gsc_clicks_march', 'ga4_sessions_march']:
    frame[col] = frame[col].fillna(0)
frame['gsc_avg_position_march'] = frame['gsc_avg_position_march'].fillna(0)
frame['gsc_impressions_march_log'] = frame['gsc_impressions_march'].apply(lambda x: 0 if x <= 0 else round(math.log1p(x), 4))
frame['gsc_clicks_march_log'] = frame['gsc_clicks_march'].apply(lambda x: 0 if x <= 0 else round(math.log1p(x), 4))
frame['ga4_sessions_march_log'] = frame['ga4_sessions_march'].apply(lambda x: 0 if x <= 0 else round(math.log1p(x), 4))

feature_frame = frame[['gsc_impressions_march_log', 'gsc_clicks_march_log', 'gsc_avg_position_march', 'ga4_sessions_march_log', 'gsc_available_march', 'future_decline_proxy']].copy()
feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gsc_impressions_march_log,gsc_clicks_march_log,gsc_avg_position_march,ga4_sessions_march_log,gsc_available_march,future_decline_proxy
0,4.3567,0.0000,4.074107,0.0,1,1
1,6.4019,1.6094,4.428747,0.0,1,0
2,6.6983,0.6931,4.866123,0.0,1,1
3,4.4188,0.0000,8.978086,0.0,1,0
4,7.5278,1.9459,1.854929,0.0,1,0


## 2. Feature notes (meaning, missing, categorical, available-when?)

- gsc_impressions_march_log: the logged March impression total; missing values are filled with 0 and it is knowable at the decision moment because it is fully observed before April starts.
- gsc_clicks_march_log: the logged March click total; missing values are filled with 0 and it is knowable at the decision moment because it is also observed in March before the decision.
- gsc_avg_position_march: the average March position; missing values are filled with 0 and it is knowable at the decision moment because the March position signal is already in the past.
- ga4_sessions_march_log: the logged March GA4 session total; missing values are filled with 0 and it is knowable at the decision moment because it is a March measurement that exists before April.
- gsc_available_march: a boolean flag that the March slice had usable GSC coverage; it is knowable at the decision moment because it is determined from March data and not from April.

In [2]:
feature_frame.describe().T[['mean','std','min','max']].round(3)

,mean,std,min,max
gsc_impressions_march_log,2.670723,3.093191,0.0,13.3328
gsc_clicks_march_log,0.361634,0.860786,0.0,8.6428
gsc_avg_position_march,8.531578,15.182674,0.0,309.0
ga4_sessions_march_log,0.454711,0.966217,0.0,7.9124
gsc_available_march,0.533246,0.498894,0.0,1.0
future_decline_proxy,0.283611,0.450751,0.0,1.0


## 3. The leakage hunt

I will add one label-derived column on purpose, show the score jump, then delete it and keep the honest score.

In [3]:
X = feature_frame[['gsc_impressions_march_log', 'gsc_clicks_march_log', 'gsc_avg_position_march', 'ga4_sessions_march_log', 'gsc_available_march']]
y = feature_frame['future_decline_proxy']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
clf = LogisticRegression(max_iter=500, random_state=42)
clf.fit(X_train, y_train)
pred = clf.predict(X_test)
honest_accuracy = accuracy_score(y_test, pred)

leaky_frame = feature_frame.copy()
leaky_frame['future_decline_proxy_leak'] = leaky_frame['future_decline_proxy']
X2 = leaky_frame[['gsc_impressions_march_log', 'gsc_clicks_march_log', 'gsc_avg_position_march', 'ga4_sessions_march_log', 'gsc_available_march', 'future_decline_proxy_leak']]
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y, test_size=0.3, random_state=42, stratify=y)
clf2 = LogisticRegression(max_iter=500, random_state=42)
clf2.fit(X2_train, y2_train)
pred2 = clf2.predict(X2_test)
leaky_accuracy = accuracy_score(y2_test, pred2)

leaky_frame = leaky_frame.drop(columns=['future_decline_proxy_leak'])

print('Honest accuracy:', round(honest_accuracy, 4))
print('With a label-derived leak:', round(leaky_accuracy, 4))
print('Columns after deleting the leak:', leaky_frame.columns.tolist())

Honest accuracy: 0.7665
With a label-derived leak: 1.0
Columns after deleting the leak: ['gsc_impressions_march_log', 'gsc_clicks_march_log', 'gsc_avg_position_march', 'ga4_sessions_march_log', 'gsc_available_march', 'future_decline_proxy']


## 4. What I excluded and why

- April metrics: they are in the future relative to the March decision point and would leak the label window.
- trend_direction-like columns: they are derived from a later outcome signal and would make the model look stronger than it really is.
- product flags: they are product decisions, not observed measurements, so they are not honest features for discovery.

In [4]:
excluded_reasons = {
    'April metrics': 'Future-window information that should not be available at the March decision point.',
    'trend_direction-like columns': 'Derived from the target concept itself and therefore label-adjacent.',
    'product flags': 'Product decisions rather than observed measurements.'
}
pd.Series(excluded_reasons)

,0
April metrics,Future-window information that should not be a...
trend_direction-like columns,Derived from the target concept itself and the...
product flags,Product decisions rather than observed measure...


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.